<div dir="rtl" style="direction:rtl;text-align:right;font:700 20px Tahoma,Arial,sans-serif;line-height:1.9">
# KhaledDSML 0.6 · المقارنات البعدية ومنحنيات ROC

## مختبر التحليل الإحصائي المتقدم — جامعة الوادي

تصميم أكاديمي احترافي باللغة العربية والإنجليزية، مع جداول مقارنات دقيقة وثيمات Aurora وOcean وAmethyst.

**تنبيه:** الإصدار 0.6 نسخة تطويرية. الأرقام ناتجة عن بيانات اصطناعية تعليمية، وليست استنتاجات عن سكان أو جامعات حقيقية.
</div>

<div dir="rtl" style="text-align:right">
## 1. التثبيت

نفّذ الخلية ثم اختر **Runtime → Restart session**.
</div>

In [ ]:
%pip install --no-cache-dir --force-reinstall --no-deps "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.6-posthoc-roc-comparison"

<div dir="rtl" style="text-align:right">
## 2. المقارنات البعدية بين ثلاث مجموعات

اختبار Games–Howell يسمح باختلاف تباينات المجموعات المستقلة، ويعرض فواصل الثقة المصححة.
</div>

In [ ]:
import numpy as np
import khaled_dsml as kd

rng = np.random.default_rng(42)
groups = {
    "الوادي": rng.normal(64, 8, size=32),
    "بسكرة": rng.normal(69, 13, size=35),
    "الجزائر": rng.normal(77, 15, size=31),
}
posthoc = kd.games_howell_comparisons(groups)

kd.display_analysis_dashboard(
    posthoc,
    title_ar="مرصد مقارنات الأداء",
    title_en="POST-HOC GROUP COMPARISONS",
    theme="aurora", font_size=22
)

<div dir="rtl" style="text-align:right">
## 3. اختبار Dunnett: مقارنة علاجين بالضبط

يُقارن كل علاج بالمجموعة الضابطة، وليس جميع العلاجات بعضها ببعض.
</div>

In [ ]:
control_comparisons = kd.dunnett_control_comparisons(
    groups, control="الوادي", seed=42
)
kd.display_analysis_dashboard(
    control_comparisons,
    title_ar="فحوص مقارنة المعالجات بالمجموعة الضابطة",
    title_en="DUNNETT · CONTROL COMPARISONS",
    theme="ocean", font_size=21
)

<div dir="rtl" style="text-align:right">
## 4. DeLong: مقارنة ROC-AUC بين نموذجين على الأشخاص أنفسهم

مقارنة التمييز فقط، لا تختبر سببية النموذج أو معايرته. التنبؤات على عينات الاختبار نفسها للحالتين.
</div>

In [ ]:
labels=np.array([0]*80+[1]*80)
scores_a=np.r_[rng.normal(0,1,80),rng.normal(1.0,1,80)]
scores_b=np.r_[rng.normal(0,1,80),rng.normal(.4,1,80)]

delong=kd.delong_auc_comparison(labels,scores_a,scores_b)
bootstrap=kd.stratified_paired_auc_bootstrap(
    labels,scores_a,scores_b,repetitions=500,seed=42
)

kd.display_analysis_dashboard(
    [delong, bootstrap],
    title_ar="لوحة تقييم النموذجين",
    title_en="CORRELATED ROC-AUC · MODEL EVALUATION",
    theme="amethyst", font_size=21
)

<div dir="rtl" style="text-align:right">
## 5. حفظ تقرير جامعي فني

اختر Print → Save as PDF بعد فتح ملف HTML الناتج. جميع المؤشرات الموجودة مأخوذة من حسابات فعلية؛ لن تُولّد قيم غير معروفة.
</div>

In [ ]:
report = kd.save_analysis_dashboard(
    [posthoc,control_comparisons,delong,bootstrap],
    "KhaledDSML_Posthoc_ROC_AR_EN.html",
    title_ar="التقرير الجامعي للمقارنات وتقييم النماذج",
    title_en="POST-HOC AND ROC MODEL COMPARISON",
    theme="aurora",
    font_size=21,
)
print("Saved:",report)

<div dir="rtl" style="direction:rtl;text-align:right;font:600 16px Tahoma,Arial,sans-serif;line-height:1.9">
## ضوابط البحث العلمي

- المقارنات المستقلة لا تصلح تلقائيًا للقياسات المكررة.
- DeLong يقارن ROC-AUC لنموذجين على نفس المستجيبين مع افتراض استقلال المستجيبين.
- لا تساوي قيمة p احتمال صحة الفرضية الصفرية.
- خوارزميات DeLong وBootstrap لا تعوّض خطة اختيار النموذج وفصل التدريب عن الاختبار.
- تبقى بنود أخرى من كتالوج الـ256 بحاجة إلى تنفيذ مستقل وتوثيق مرجعي.
</div>